# Sovereign Byte Firewall — Kaggle GPU Training (v2 masking)

Retrain with the fixed masking scheme (TLS continuation, stochastic header fields, QUIC).

**Rules baked into this notebook — do not undo them:**
1. **Train on the MONDAY file only** (benign-only day). Wednesday is an ATTACK day: training on it contaminates the model AND it is the held-out evaluation day for `evaluate_cic_days.py`.
2. **1 epoch.** Repeating epochs over the same file caused real regression (32% -> ~6% held-out detection).
3. **Plain CrossEntropy.** FocalLoss proven worse.
4. **Fresh start.** Old checkpoints were trained under the old masking — not resumable. Checkpoints push to a NEW HF repo to avoid gs-number collisions.

Make sure GPU accelerator is ON: Settings > Accelerator > GPU T4 x2 (Tesla P100 is NOT compatible with the pre-installed PyTorch version due to compute capability sm_60 requirements).

In [ ]:
import subprocess, sys
for pkg in ['scapy', 'huggingface_hub>=0.25', 'accelerate']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)
print('Dependencies installed')

In [ ]:
import os, subprocess
REPO_DIR = '/kaggle/working/sovereign-byte-firewall'
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/TheSPST/sovereign-byte-firewall.git', REPO_DIR], check=True)
os.chdir(REPO_DIR)
# Sanity: this run REQUIRES the v2 masking commits
log = subprocess.run(['git', 'log', '--oneline', '-8'], capture_output=True, text=True).stdout
print(log)
assert 'stochastic header fields' in subprocess.run(['git', 'log', '--format=%s', '-30'], capture_output=True, text=True).stdout, \
    'v2 masking commits missing — did you push c59df36+ to GitHub?'
print('Repo ready (v2 masking confirmed) at:', os.getcwd())

In [ ]:
import os
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print('Logged in via Kaggle Secret')
except Exception:
    HF_TOKEN = ''  # paste token here as fallback
    if HF_TOKEN:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print('Logged in via token')
    else:
        print('WARNING: No HF token - checkpoints will NOT be uploaded')
# NEW repo: v2-masking checkpoints must not mix with old-masking gs-numbers
os.environ['HF_REPO_ID'] = 'spst01/sovereign-byte-firewall-v2-masking'
print('Checkpoint backup repo:', os.environ['HF_REPO_ID'])

In [ ]:
import os
from huggingface_hub import hf_hub_download

# TRAIN ON MONDAY (benign-only) — NEVER Wednesday (attack day + eval day).
# Source: bvsam/cic-ids-2017 (public, non-gated) — canonical CIC-IDS2017 raw
# pcaps. Same repo also holds pcap/Wednesday-workingHours.pcap for the
# evaluate_cic_days.py eval later.
DATASET_PATH = None

# 1) Manually-attached Kaggle datasets first (faster if you cached it)
for root, _, files in os.walk('/kaggle/input'):
    for f in files:
        if 'onday' in f and f.endswith('.pcap'):
            DATASET_PATH = os.path.join(root, f)
            print('Monday pcap found (Kaggle input):', DATASET_PATH)
            break
    if DATASET_PATH:
        break

# 2) Otherwise check if already downloaded or pull Monday from the public CIC-IDS2017 HF dataset (~10 GB)
if not DATASET_PATH:
    possible_local = '/kaggle/working/pcap/Monday-WorkingHours.pcap'
    if os.path.exists(possible_local) and os.path.getsize(possible_local) > 10 * 1e9:
        DATASET_PATH = possible_local
        print('Monday pcap already exists locally on disk: (skipping download)', DATASET_PATH)
    else:
        print('Downloading Monday-WorkingHours.pcap from bvsam/cic-ids-2017 (~10 GB, one-time)...... ')
        DATASET_PATH = hf_hub_download(repo_id='bvsam/cic-ids-2017',
                                       filename='pcap/Monday-WorkingHours.pcap',
                                       repo_type='dataset', local_dir='/kaggle/working/')
        print('Monday pcap downloaded:', DATASET_PATH)

assert 'ednesday' not in os.path.basename(DATASET_PATH), 'Refusing to train on Wednesday (attack/eval day)!'
assert 'onday' in os.path.basename(DATASET_PATH), 'Training file is not the Monday benign capture!'
print(f'Size: {os.path.getsize(DATASET_PATH)/1e9:.2f} GB')


In [ ]:
import torch, subprocess
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected! Turn on GPU accelerator in Settings.')
major, minor = torch.cuda.get_device_capability(0)
if major < 7:
    raise RuntimeError(f'Detected GPU {torch.cuda.get_device_name(0)} with CUDA capability {major}.{minor}. ' 
                       f'The installed PyTorch version requires compute capability >= 7.0 (sm_70+). ' 
                       f'Tesla P100 (sm_60) is NOT compatible. Please switch your Kaggle accelerator to "GPU T4 x2".')
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'GPU[{i}]: {name} - {vram:.1f}GB VRAM')
subprocess.run(['nvidia-smi'])


In [ ]:
import subprocess, os

# Find the attached eval dataset: any /kaggle/input folder with a benign
# calibration pcap (normal.pcap) AND a held-out attack pcap (0day.pcap).
# normal2.pcap (benign holdout) is OPTIONAL — the watcher reuses normal.pcap
# if it's absent, so we don't require it here.
eval_dir = None
for root, _, files in os.walk('/kaggle/input'):
    if 'normal.pcap' in files and '0day.pcap' in files:
        eval_dir = root
        break

if eval_dir:
    print('Eval dataset dir:', eval_dir)
    print('Contents:', sorted(f for f in os.listdir(eval_dir) if f.endswith('.pcap')))
    env = dict(os.environ, CUDA_VISIBLE_DEVICES='')   # CPU only — no GPU contention
    if 'hf_token' in globals() and hf_token:
        env['HF_TOKEN'] = hf_token
    proc = subprocess.Popen(
        ['python', 'scripts/kaggle_eval_watcher.py',
         '--checkpoints_dir', '/kaggle/working/checkpoints',
         '--eval_data_dir', eval_dir,
         '--max_sequence_length', '512'],
        stdout=open('/kaggle/working/eval_watcher.log', 'w'),
        stderr=subprocess.STDOUT, env=env, start_new_session=True)
    print('Eval watcher started (PID', proc.pid, ') -> /kaggle/working/eval_watcher.log')
    # Give preflight a moment, then show its verdict inline so you see it immediately
    import time; time.sleep(12)
    print('--- eval_watcher.log (first lines) ---')
    print(open('/kaggle/working/eval_watcher.log').read())
else:
    print('WARNING: no eval dataset found in /kaggle/input (need a folder with normal.pcap + 0day.pcap).')
    print('Attach your eval PCAPs as a Kaggle dataset to enable live per-checkpoint evaluation.')
    print('Training will run fine without it; use evaluate_cic_days.py on the final checkpoint instead.')


In [ ]:
import sys, os, re, shutil
from huggingface_hub import HfApi, hf_hub_download
sys.path.insert(0, '/kaggle/working/sovereign-byte-firewall')
os.chdir('/kaggle/working/sovereign-byte-firewall')

# --- Auto-restore latest checkpoint from HF ---
repo_id = os.environ.get('HF_REPO_ID', 'spst01/sovereign-byte-firewall-v2-masking')
local_ckpt_dir = '/kaggle/working/checkpoints'
local_ckpt_path = os.path.join(local_ckpt_dir, 'latest_patcher.pt')
os.makedirs(local_ckpt_dir, exist_ok=True)

try:
    api = HfApi()
    files = api.list_repo_files(repo_id=repo_id)
    checkpoint_files = [f for f in files if f.startswith('checkpoints/latest_patcher_') and f.endswith('.pt')]
    
    if checkpoint_files:
        def get_step(filename):
            match = re.search(r'_gs(\d+)_', filename)
            return int(match.group(1)) if match else -1
        latest_file = max(checkpoint_files, key=get_step)
        latest_step = get_step(latest_file)
        
        print(f'[CheckpointRestorer] Found latest checkpoint on HF: {latest_file} (Step {latest_step})')
        print('[CheckpointRestorer] Downloading checkpoint file...')
        downloaded = hf_hub_download(repo_id=repo_id, filename=latest_file, local_dir='/tmp/hf_ckpt')
        shutil.copy(downloaded, local_ckpt_path)
        print(f'[CheckpointRestorer] Successfully restored to: {local_ckpt_path}')
    else:
        print('[CheckpointRestorer] No checkpoint files found on HF repo. Starting training from scratch.')
except Exception as e:
    print(f'[CheckpointRestorer] Could not auto-restore checkpoint from HF (will start from scratch/local): {e}')

sys.argv = [
    'run_kaggle.py',
    '--dataset_path', DATASET_PATH,
    '--epochs', '1',                 # >1 epoch = proven regression
    '--batch_size', '32',
    '--max_sequence_length', '512',
    '--lr', '5e-4',
    '--use_focal_loss', 'false',     # FocalLoss proven worse
    '--checkpoints_dir', '/kaggle/working/checkpoints',
    '--allow_resume',                # Allow resuming training from the downloaded checkpoint
    # '--total_steps', '1000000',    # uncomment with the known windows/epoch for an exact LR schedule
]

from run_kaggle import main
main()


In [ ]:
import glob, os
ckpts = glob.glob('/kaggle/working/checkpoints/*.pt')
print(f'{len(ckpts)} checkpoint(s) saved:')
for c in sorted(ckpts):
    print(f'  {os.path.basename(c)} ({os.path.getsize(c)/1e6:.1f} MB)')
print('View on HF Hub: https://huggingface.co/spst01/sovereign-byte-firewall-v2-masking')
print('Monitor from your Mac: python scripts/aikosh_monitor.py --hf_repo spst01/sovereign-byte-firewall-v2-masking')